In [36]:
import numpy as np

In [76]:
images = [f"img_{i}" for i in range(4)]
prompts = [f"prompt_{i}" for i in range(4)]
metadatas = [{"info": f"meta_{i}"} for i in range(4)]
final_scores = []
for prompt, image, metadata in zip(prompts, images, metadatas):
    sub_images = [f"{image}_{i}" for i in range(4)]

    dimension_scores : dict[str, float] = {
        "Style": 0.0,
        "Identity": 0.0,
        "Logic": 0.0
    }
    # Compute scores for each prompt-image pair from different dimensions
    for dimension in ["Style", "Identity", "Logic"]:
        # Get criteria for this dimension
        criterion_scores = [] # each item is a list of scores, each score is for one criterion
        # Compute each pair of neighbors
        for i in range(len(sub_images) - 1):
            for j in range(i + 1, len(sub_images)):
                img1 = sub_images[i]
                img2 = sub_images[j]

                scores = np.random.rand(3).tolist()  # Simulate scores for three criteria

                criterion_scores.append(scores)

        # Transpose dimension_scores, to make each item is a list of scores for each criterion
        criterion_scores = list(map(list, zip(*criterion_scores)))
        # Compute the average score within each criterion
        criterion_scores = [sum(scores) / len(scores) if scores else 0.0 for scores in criterion_scores]

        # Compute the overall score for this dimension
        overall_score = sum(criterion_scores) / len(criterion_scores) if criterion_scores else 0.0
        dimension_scores[dimension] = overall_score
    
    # Compute average score for this prompt-image
    final_scores.append(sum(dimension_scores.values()) / len(dimension_scores))



# Compute average scores for each dimension
final_scores

[0.4588569650214785,
 0.5352985558103803,
 0.48682277197665025,
 0.5441325581563561]

In [57]:
final_scores.mean(axis=(1,2)).tolist()

[0.565802553712208, 0.518131963258245, 0.4696826756002077, 0.456097489858642]

In [11]:
import json
import re
prompt_file_path = 'flow_grpo/dataset/T2IS/prompt_train.json'

In [12]:
def extract_grid_info(prompt) -> tuple[int, int]:
    # Grid can be represented as int x int, or int ⨉ int. ⨉ has unicode \u2a09
    match = re.findall(r'(\d+)\s*[x⨉]\s*(\d+)', prompt)
    if len(match) == 0:
        return (1, 1)

    return (int(match[0][0]), int(match[0][1]))

In [2]:
data = json.load(open(prompt_file_path))

In [13]:
# Group data by its grid form
groups = {}
for item in data:
    grid_info = extract_grid_info(item['prompt'])
    if grid_info not in groups:
        groups[grid_info] = []
    
    groups[grid_info].append(item)


In [16]:
{k: len(v) for k, v in groups.items()}

{(1, 3): 112,
 (2, 2): 285,
 (1, 5): 116,
 (2, 3): 68,
 (1, 2): 7,
 (1, 7): 5,
 (2, 4): 3}

In [18]:
square_2x2 = groups[(2,2)]

In [27]:
# Split the data into train and test sets
train_data = square_2x2[:-64]
test_data = square_2x2[-64:]

In [30]:
with open('flow_grpo/dataset/T2IS/train_metadata.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open('flow_grpo/dataset/T2IS/test_metadata.jsonl', 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')